# Sessions & State

Remember things across turns with session state.

> Part of the **ADK Katas**. Work top-to-bottom: read the concept, fill in the
> `# TODO`s in the starter cell, then run the **Check** cell — it grades your
> code offline (no API call). The **Run it live** cell needs a `GOOGLE_API_KEY`.


## Kata 04 — Sessions & State

Agents remember things across turns via **session state** — a dict scoped to
the conversation. A tool can read/write it by accepting a `ToolContext`
parameter (ADK injects it; the model never sees it):

```python
from google.adk.tools.tool_context import ToolContext

def remember_city(city: str, tool_context: ToolContext) -> dict:
    tool_context.state["favorite_city"] = city
    return {"status": "success"}
```

Setting `output_key="last_reply"` on the agent also stores its final text in
`state["last_reply"]` automatically.

## Your task

1. Implement `remember_city(city, tool_context)` to save the city into
   `tool_context.state["favorite_city"]` and return a success dict.
2. Build `root_agent` (name `"memory_bot"`) with that tool and with
   `output_key="last_reply"`.

In [ ]:
# Setup — run me first
import sys, pathlib
# Make the kata library importable whether opened from adk-katas/ or adk-katas/notebooks/
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "kata_helpers.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from kata_helpers import *          # load_env, has_api_key, run_agent, check, grade, requires_key
from kata_content import BY_SLUG

KATA = BY_SLUG["sessions-state"]
load_env()
print("API key configured:" , has_api_key())


In [ ]:
# ✏️ Your code — fill in the TODOs
from google.adk.agents import Agent
from google.adk.tools.tool_context import ToolContext

def remember_city(city: str, tool_context: ToolContext) -> dict:
    """Saves the user's favourite city to session state.

    Args:
        city: The city to remember.
    """
    # TODO: write the city into tool_context.state["favorite_city"]
    return {}

# TODO: build root_agent with remember_city and output_key="last_reply".
root_agent = None

In [ ]:
# ✅ Check your work (offline — grades the symbols you defined above)
results = KATA.run_checks(globals())
grade([check(r.label, r.passed, r.detail) for r in results])


In [ ]:
# ▶️ Run it live (needs GOOGLE_API_KEY). Each run is a fresh single-turn session.
agent = globals().get(KATA.chat_symbol)

if not KATA.chattable:
    print("This kata has no chat agent — the Check cell is the goal. 🎯")
elif has_api_key() and agent is not None:
    result = await run_agent(agent, 'My favourite city is Tokyo.')   # noqa: F704  (top-level await is fine in Jupyter)
    print("Response:", result.text)
    if result.tool_calls:
        print("Tools called:", result.tool_calls, result.tool_args)
    if result.transfers:
        print("Transferred to:", result.transfers)
    if result.state:
        print("Session state:", result.state)
else:
    requires_key(lambda: None)


---
### Stuck? Reveal the reference solution

<details>
<summary>Show solution</summary>

```python
from google.adk.agents import Agent
from google.adk.tools.tool_context import ToolContext

def remember_city(city: str, tool_context: ToolContext) -> dict:
    """Saves the user's favourite city to session state.

    Args:
        city: The city to remember.
    """
    tool_context.state["favorite_city"] = city
    return {"status": "success", "saved": city}

root_agent = Agent(
    name="memory_bot",
    model="gemini-2.5-flash",
    instruction="Remember the user's favourite city with remember_city.",
    tools=[remember_city],
    output_key="last_reply",
)
```

</details>

When you're done, try the same kata in the React app's live chat (`./dev.sh`
from the repo root) to watch the tool-call traces.
